# 03 · Mapa Autoorganizado (SOM) y análisis topológico
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Valoo-1201/vfc-caos-som/blob/main/notebooks/03_SOM_agrupamiento.ipynb)

**Responsable:** Integrante C · **Entrada:** `results/features.csv` · **Salida:** figuras del SOM y `results/som_resumen.csv`

El SOM proyecta el espacio de métricas (13 dimensiones) en una malla 2D **preservando la topología**: ventanas parecidas caen en neuronas vecinas. Herramientas de lectura:
- **U-matriz**: distancia entre neuronas vecinas; las crestas altas son fronteras entre grupos.
- **Planos de componentes**: valor de cada métrica sobre el mapa; explican *por qué* se separan los grupos.
- **Mapa de impactos**: dónde caen las ventanas de cada grupo clínico.

In [ ]:
#@title 0 · Configuración (ejecutar siempre primero)
REPO = "Valoo-1201/vfc-caos-som"   # <- usuario/repositorio en GitHub
RAMA = "main"
USAR_DRIVE = True                  # guarda las series R-R en Google Drive para no descargarlas otra vez
MODO_DEMO = False                  # True = datos sintéticos (solo para probar el código sin internet)

import os, sys, subprocess
EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    os.chdir("/content")
    token = None
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")      # Colab -> 🔑 Secretos (solo hace falta si el repo es privado)
    except Exception:
        pass
    url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
    if not os.path.exists("/content/vfc-caos-som"):
        subprocess.run(["git", "clone", "-q", "-b", RAMA, url, "/content/vfc-caos-som"], check=True)
    else:
        subprocess.run(["git", "-C", "/content/vfc-caos-som", "pull", "-q"], check=False)
    os.chdir("/content/vfc-caos-som")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    if USAR_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        DATA_DIR = "/content/drive/MyDrive/vfc-caos-som-data"
    else:
        DATA_DIR = "/content/data"
else:  # ejecución local: el notebook está en notebooks/
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")
    DATA_DIR = "data"

sys.path[:0] = [os.path.abspath("src"), os.getcwd()]
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs("results/figures", exist_ok=True)

import numpy as np, pandas as pd, matplotlib.pyplot as plt
SEED = 42
np.random.seed(SEED)
pd.set_option("display.precision", 3)
print("Repositorio:", os.getcwd(), "| Datos:", DATA_DIR, "| Demo:", MODO_DEMO)

In [ ]:
from sklearn.preprocessing import StandardScaler
from hrvchaos import metrics
from hrvchaos.som import SOM
from hrvchaos.evaluate import som_clusters, cluster_agreement
from hrvchaos.plots import plot_u_matrix_with_hits, plot_component_planes, hex_plot

if MODO_DEMO and not os.path.exists("results/features.csv"):
    raise FileNotFoundError("Ejecuta antes el notebook 02 (también en modo demo).")
features = pd.read_csv("results/features.csv")

GRUPOS = ["RSN", "ICC"]          # eje principal; para envejecimiento usar ["Joven", "Mayor"]
FEATS = metrics.LINEAR + metrics.NONLINEAR
d = features[features.group.isin(GRUPOS)].dropna(subset=FEATS).reset_index(drop=True)
X = StandardScaler().fit_transform(d[FEATS].values)
y = d.group.values
print(X.shape, dict(zip(*np.unique(y, return_counts=True))))

## Entrenamiento
Tamaño de malla con la heurística de Vesanto (≈ 5·√N neuronas). Inicialización PCA, σ y tasa de aprendizaje decrecientes.

In [ ]:
filas, cols = SOM.suggest_shape(len(X), X)
som = SOM(filas, cols, X.shape[1], seed=SEED).fit(X, n_iter=20000, verbose=True)
print(f"Malla {filas}x{cols} | QE = {som.quantization_error(X):.3f} | TE = {som.topographic_error(X):.3f}")

In [ ]:
plot_u_matrix_with_hits(som, X, y)
plt.savefig("results/figures/03_umatriz.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
fig = plot_component_planes(som, FEATS)
fig.savefig("results/figures/03_planos_componentes.png", dpi=150); plt.show()

## Agrupamiento en dos niveles
k-means sobre los prototipos del SOM (Vesanto & Alhoniemi, 2000). El *Adjusted Rand Index* (ARI) compara los grupos encontrados **sin supervisión** con la etiqueta clínica: 1 = coinciden, 0 = azar.

In [ ]:
resultados = []
for k in [2, 3, 4]:
    cl = som_clusters(som, X, k=k, seed=SEED)
    resultados.append({"k": k, "ARI": cluster_agreement(y, cl)})
    if k == 2:
        fig, ax = plt.subplots(figsize=(6, 5))
        hex_plot(ax, som, som.node_clusters, cmap="Set2", title="Grupos k-means (k = 2) sobre el SOM", colorbar=False)
        plt.savefig("results/figures/03_kmeans_k2.png", dpi=150); plt.show()
        print(pd.crosstab(pd.Series(y, name="clínico"), pd.Series(cl, name="cluster")))
pd.DataFrame(resultados)

## Estabilidad
El SOM depende de la semilla. Repetimos con 5 semillas: QE, TE y ARI deberían variar poco.

In [ ]:
est = []
for s in range(5):
    m = SOM(filas, cols, X.shape[1], seed=s).fit(X, n_iter=20000)
    est.append({"seed": s, "QE": m.quantization_error(X), "TE": m.topographic_error(X),
                "ARI_k2": cluster_agreement(y, som_clusters(m, X, k=2, seed=s))})
est = pd.DataFrame(est)
est.to_csv("results/som_resumen.csv", index=False)
est.describe().loc[["mean", "std"]]